# Data Cleaning

### Data Cleaning Plan

| Task | Action |
|---|---|
| Naming standardization | Rename column to match snake_case convention |
| Exact duplicates | Remove 1,385 duplicated rows |
| Athlete-event duplicates | Investigate before removing anything |
| Missing Medal | Preserve (represents no medal) |
| Missing Age | Preserve initially (unknown age) |
| Missing Height/Weight | Preserve initially (unknown measurements) |
| Categorical values | Standardize and validate |
| NOC codes | Merge with `noc_regions.csv` and investigate unmatched codes |
| Numerical values | Investigate suspicious values and handle confirmed errors |
| Final validation | Verify the cleaned dataset |

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
athletes_df = pd.read_csv('../data/raw/athlete_events.csv')
noc_df = pd.read_csv('../data/raw/noc_regions.csv')

### 1) Naming standardization

In [3]:
athletes_df.columns

Index(['ID', 'Name', 'Sex', 'Age', 'Height', 'Weight', 'Team', 'NOC', 'Games',
       'Year', 'Season', 'City', 'Sport', 'Event', 'Medal'],
      dtype='str')

In [4]:
athletes_df.columns = (
    athletes_df.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_')
)
athletes_df.columns

Index(['id', 'name', 'sex', 'age', 'height', 'weight', 'team', 'noc', 'games',
       'year', 'season', 'city', 'sport', 'event', 'medal'],
      dtype='str')

### 2) Remove exact duplicates

In [5]:
athletes_df.duplicated().sum()

np.int64(1385)

In [6]:
athletes_df = athletes_df.drop_duplicates().reset_index(drop=True)

In [7]:
athletes_df.duplicated().sum()

np.int64(0)

In [8]:
athletes_df.shape

(269731, 15)

We're only removing completely identical rows. We are NOT removing repeated athlete-event combinations yet, because those require separate investigation.

### 3) Investigate Athlete-Event Duplicates.

In [9]:
athletes_df.duplicated(subset=['id', 'games', 'event']).sum()

np.int64(95)

In [10]:
athlete_event_duplicates = athletes_df[
    athletes_df.duplicated(
        subset=['id', 'games', 'event'],
        keep=False
    )
].sort_values(['id', 'games', 'event'])

athlete_event_duplicates

,id,name,sex,age,height,weight,team,noc,games,year,season,city,sport,event,medal
4016,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Silver
4017,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
10891,5959,Louis Albert Auguste-Dormeuil (Dormeuil-),M,31.0,NaN,NaN,Pettit-Poucet-6,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,NaN
10892,5959,Louis Albert Auguste-Dormeuil (Dormeuil-),M,31.0,NaN,NaN,Carabinier-15,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Gold
16074,8683,Jacques Baudrier,M,28.0,NaN,NaN,Crabe II-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Silver
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251416,126533,Franois Antoine Vilamitjana (Caubet-),M,53.0,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
256789,129244,mile Robert Wegelin,M,24.0,NaN,NaN,Club Nautique de Lyon-2,FRA,1900 Summer,1900,Summer,Paris,Rowing,Rowing Men's Coxed Fours,Silver
256790,129244,mile Robert Wegelin,M,24.0,NaN,NaN,Club Nautique de Lyon-2,FRA,1900 Summer,1900,Summer,Paris,Rowing,Rowing Men's Coxed Fours,NaN
263921,132751,Jack Butler Yeats,M,52.0,NaN,NaN,Ireland,IRL,1924 Summer,1924,Summer,Paris,Art Competitions,Art Competitions Mixed Painting,Silver


In [11]:
athlete_event_duplicates.groupby(
    ['id', 'games', 'event']
).size().sort_values(ascending=False)

id      games        event                                        
119448  1900 Summer  Sailing Mixed Open                               4
119449  1900 Summer  Sailing Mixed Open                               4
67492   1900 Summer  Sailing Mixed 0.5-1 Ton                          3
28501   1948 Summer  Art Competitions Mixed Painting, Applied Arts    3
5959    1900 Summer  Sailing Mixed 0.5-1 Ton                          2
                                                                     ..
124464  1900 Summer  Sailing Mixed 3-10 Ton                           2
125022  1900 Summer  Equestrianism Mixed Four-In-Hand Competition     2
126533  1900 Summer  Sailing Mixed 1-2 Ton                            2
129244  1900 Summer  Rowing Men's Coxed Fours                         2
132751  1924 Summer  Art Competitions Mixed Painting                  2
Length: 89, dtype: int64

In [12]:
athlete_event_duplicates[athlete_event_duplicates['id'] == 28501]

,id,name,sex,age,height,weight,team,noc,games,year,season,city,sport,event,medal
55305,28501,Alex Walter Diggelmann,M,33.0,NaN,NaN,Switzerland,SUI,1936 Summer,1936,Summer,Berlin,Art Competitions,"Art Competitions Mixed Painting, Applied Arts",Gold
55306,28501,Alex Walter Diggelmann,M,33.0,NaN,NaN,Switzerland,SUI,1936 Summer,1936,Summer,Berlin,Art Competitions,"Art Competitions Mixed Painting, Applied Arts",NaN
55308,28501,Alex Walter Diggelmann,M,45.0,NaN,NaN,Switzerland,SUI,1948 Summer,1948,Summer,London,Art Competitions,"Art Competitions Mixed Painting, Applied Arts",Silver
55309,28501,Alex Walter Diggelmann,M,45.0,NaN,NaN,Switzerland,SUI,1948 Summer,1948,Summer,London,Art Competitions,"Art Competitions Mixed Painting, Applied Arts",Bronze
55310,28501,Alex Walter Diggelmann,M,45.0,NaN,NaN,Switzerland,SUI,1948 Summer,1948,Summer,London,Art Competitions,"Art Competitions Mixed Painting, Applied Arts",NaN


In [13]:
#distinct teams per athelete
team_counts = (
    athlete_event_duplicates
    .groupby(['id', 'games', 'event'])['team']
    .nunique()
)

team_counts.value_counts().sort_index()

team
1    69
2    17
3     1
4     2
Name: count, dtype: int64

In [14]:
#distinct medals per athelete-event
medal_counts = (
    athlete_event_duplicates
    .groupby(['id', 'games', 'event'])['medal']
    .nunique()
)

medal_counts.value_counts().sort_index()

medal
0     9
1    59
2    21
Name: count, dtype: int64

In [15]:
duplicate_summary = (
    athlete_event_duplicates
    .groupby(['id', 'games', 'event'])
    .agg(
        record_count=('id', 'size'),
        team_count=('team', 'nunique'),
        medal_count=('medal', 'nunique'),
        teams=('team', lambda x: list(x.unique())),
        medals=('medal', lambda x: list(x.unique()))
    )
    .reset_index()
)
multiple_teams_medals = duplicate_summary[
    (duplicate_summary['team_count'] > 1) &
    (duplicate_summary['medal_count'] > 1)
]

multiple_teams_medals

,id,games,event,record_count,team_count,medal_count,teams,medals
2,8683,1900 Summer,Sailing Mixed 0.5-1 Ton,2,2,2,"[Crabe II-1, Crabe II-4]","[Silver, Bronze]"
49,67492,1900 Summer,Sailing Mixed 0.5-1 Ton,3,3,2,"[Crabe II-1, Crocodile-11, Crabe II-4]","[Silver, nan, Bronze]"
54,74870,1900 Summer,Sailing Mixed 0.5-1 Ton,2,2,2,"[Crabe II-1, Crabe II-4]","[Silver, Bronze]"
56,75648,1900 Summer,Sailing Mixed 0.5-1 Ton,2,2,2,"[Crabe II-1, Crabe II-4]","[Silver, Bronze]"
59,78807,1900 Summer,Sailing Mixed 0.5-1 Ton,2,2,2,"[Scamasaxe-2, Scamasaxe-3]","[Bronze, Silver]"
60,79412,1900 Summer,Sailing Mixed 0.5-1 Ton,2,2,2,"[Scamasaxe-2, Scamasaxe-3]","[Bronze, Silver]"
82,124115,1900 Summer,Sailing Mixed 0.5-1 Ton,2,2,2,"[Crabe II-1, Crabe II-4]","[Silver, Bronze]"


In [16]:
cases = multiple_teams_medals[
    ['id', 'games', 'event']
]

multi_team_medal_records = athlete_event_duplicates.merge(
    cases,
    on=['id', 'games', 'event'],
    how='inner'
)

multi_team_medal_records.sort_values(
    ['id', 'games', 'event']
)

,id,name,sex,age,height,weight,team,noc,games,year,season,city,sport,event,medal
0,8683,Jacques Baudrier,M,28.0,NaN,NaN,Crabe II-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Silver
1,8683,Jacques Baudrier,M,28.0,NaN,NaN,Crabe II-4,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Bronze
2,67492,Gabriel Charles Jean Le Bret,M,27.0,NaN,NaN,Crabe II-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Silver
3,67492,Gabriel Charles Jean Le Bret,M,27.0,NaN,NaN,Crocodile-11,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,NaN
4,67492,Gabriel Charles Jean Le Bret,M,27.0,NaN,NaN,Crabe II-4,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Bronze
5,74870,Flix mile Marcotte,M,34.0,NaN,NaN,Crabe II-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Silver
6,74870,Flix mile Marcotte,M,34.0,NaN,NaN,Crabe II-4,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Bronze
7,75648,Charles William Martin,M,71.0,NaN,NaN,Crabe II-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Silver
8,75648,Charles William Martin,M,71.0,NaN,NaN,Crabe II-4,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Bronze
9,78807,Marcel Meran,M,NaN,NaN,NaN,Scamasaxe-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 0.5-1 Ton,Bronze


In [17]:
same_team_duplicates = (
    athlete_event_duplicates
    .groupby(['id', 'games', 'event', 'team'])
    .filter(lambda x: len(x) > 1)
    .sort_values(['id', 'games', 'event', 'team'])
)

same_team_duplicates

,id,name,sex,age,height,weight,team,noc,games,year,season,city,sport,event,medal
4016,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Silver
4017,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
16076,8683,Jacques Baudrier,M,28.0,NaN,NaN,Nina Claire-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
16077,8683,Jacques Baudrier,M,28.0,NaN,NaN,Nina Claire-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,NaN
16079,8684,Lucien Alexandre Baudrier,M,38.0,NaN,NaN,Nina Claire-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251416,126533,Franois Antoine Vilamitjana (Caubet-),M,53.0,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
256789,129244,mile Robert Wegelin,M,24.0,NaN,NaN,Club Nautique de Lyon-2,FRA,1900 Summer,1900,Summer,Paris,Rowing,Rowing Men's Coxed Fours,Silver
256790,129244,mile Robert Wegelin,M,24.0,NaN,NaN,Club Nautique de Lyon-2,FRA,1900 Summer,1900,Summer,Paris,Rowing,Rowing Men's Coxed Fours,NaN
263921,132751,Jack Butler Yeats,M,52.0,NaN,NaN,Ireland,IRL,1924 Summer,1924,Summer,Paris,Art Competitions,Art Competitions Mixed Painting,Silver


In [18]:
same_team_duplicates = (
    athlete_event_duplicates
    .groupby(['id', 'games', 'event', 'team'])
    .filter(lambda x: len(x) > 1)
    .sort_values(['id', 'games', 'event', 'team'])
)

same_team_duplicates

,id,name,sex,age,height,weight,team,noc,games,year,season,city,sport,event,medal
4016,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Silver
4017,2317,Auguste Albert,M,NaN,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
16076,8683,Jacques Baudrier,M,28.0,NaN,NaN,Nina Claire-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
16077,8683,Jacques Baudrier,M,28.0,NaN,NaN,Nina Claire-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,NaN
16079,8684,Lucien Alexandre Baudrier,M,38.0,NaN,NaN,Nina Claire-2,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
251416,126533,Franois Antoine Vilamitjana (Caubet-),M,53.0,NaN,NaN,Martha-1,FRA,1900 Summer,1900,Summer,Paris,Sailing,Sailing Mixed 1-2 Ton,Bronze
256789,129244,mile Robert Wegelin,M,24.0,NaN,NaN,Club Nautique de Lyon-2,FRA,1900 Summer,1900,Summer,Paris,Rowing,Rowing Men's Coxed Fours,Silver
256790,129244,mile Robert Wegelin,M,24.0,NaN,NaN,Club Nautique de Lyon-2,FRA,1900 Summer,1900,Summer,Paris,Rowing,Rowing Men's Coxed Fours,NaN
263921,132751,Jack Butler Yeats,M,52.0,NaN,NaN,Ireland,IRL,1924 Summer,1924,Summer,Paris,Art Competitions,Art Competitions Mixed Painting,Silver


In [19]:
medal_conflicts = (
    same_team_duplicates
    .groupby(['id', 'games', 'event', 'team'])
    .agg(
        record_count=('medal', 'size'),
        distinct_medals=('medal', 'nunique'),
        medals=('medal', lambda x: list(x.unique()))
    )
    .reset_index()
)
conflicting_medals = medal_conflicts[
    medal_conflicts['distinct_medals'] > 1
]

conflicting_medals

,id,games,event,team,record_count,distinct_medals,medals
0,2317,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,2,2,"[Silver, Bronze]"
10,26557,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,2,2,"[Gold, Silver]"
11,26559,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,2,2,"[Gold, Silver]"
12,26560,1900 Summer,Sailing Mixed 1-2 Ton,Lerina,2,2,"[Gold, Silver]"
16,28501,1948 Summer,"Art Competitions Mixed Painting, Applied Arts",Switzerland,3,2,"[Silver, Bronze, nan]"
19,30274,1900 Summer,Sailing Mixed 3-10 Ton,Gitana-2,2,2,"[Bronze, Silver]"
20,30285,1900 Summer,Sailing Mixed 3-10 Ton,Gitana-2,2,2,"[Bronze, Silver]"
23,30927,1900 Summer,Sailing Mixed 1-2 Ton,Martha-1,2,2,"[Silver, Bronze]"
25,39839,1900 Summer,Sailing Mixed 0-0.5 Ton,Baby-1,2,2,"[Gold, Bronze]"
28,43650,1900 Summer,Sailing Mixed 3-10 Ton,Gitana-2,2,2,"[Bronze, Silver]"


In [20]:
medal_nan_cases = medal_conflicts[
    medal_conflicts['distinct_medals'] == 1
]

medal_nan_cases

,id,games,event,team,record_count,distinct_medals,medals
1,8683,1900 Summer,Sailing Mixed 1-2 Ton,Nina Claire-2,2,1,"[Bronze, nan]"
2,8684,1900 Summer,Sailing Mixed 1-2 Ton,Nina Claire-2,2,1,"[Bronze, nan]"
3,13957,1900 Summer,Rowing Men's Coxed Fours,Cercle de l'Aviron Roubaix-4,2,1,"[Gold, nan]"
4,17308,1900 Summer,Sailing Mixed 0-0.5 Ton,Sarcelle-3,2,1,"[Bronze, nan]"
5,18617,1900 Summer,Rowing Men's Coxed Fours,Favorite Hammonia-3,2,1,"[Bronze, nan]"
6,19094,1900 Summer,Rowing Men's Coxed Fours,Cercle de l'Aviron Roubaix-4,2,1,"[Gold, nan]"
7,19917,1900 Summer,Rowing Men's Coxed Fours,Cercle de l'Aviron Roubaix-4,2,1,"[Gold, nan]"
8,24399,1900 Summer,Sailing Mixed 0.5-1 Ton,Scotia,2,1,"[Gold, nan]"
9,26058,1900 Summer,Sailing Mixed 2-3 Ton,Gwendoline-2,2,1,"[Bronze, nan]"
13,27223,1900 Summer,Rowing Men's Coxed Fours,Cercle de l'Aviron Roubaix-4,2,1,"[Gold, nan]"


In [21]:
medal_nan_groups = medal_conflicts[
    (medal_conflicts['distinct_medals'] == 1) &
    (medal_conflicts['medals'].apply(lambda x: any(pd.isna(m) for m in x)))
]

all_missing_groups = medal_conflicts[
    medal_conflicts['distinct_medals'] == 0
]

print("Conflicting medal groups:", len(conflicting_medals))
print("Medal + NaN groups:", len(medal_nan_groups))
print("All missing medal groups:", len(all_missing_groups))

Conflicting medal groups: 14
Medal + NaN groups: 55
All missing medal groups: 0


#### Athlete-event duplicates:
 Repeated combinations of athlete ID, Games, event, and team were investigated. The investigation identified groups with conflicting medal values and groups containing both medal and missing values. Since these records may reflect historical competition structures or source-data inconsistencies, they were preserved rather than removed without sufficient evidence.

### 4) Missing values

In [22]:
missing_summary = pd.DataFrame({
    'missing_count': athletes_df.isna().sum(),
    'missing_percentage': athletes_df.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary['missing_count'] > 0
].sort_values('missing_percentage', ascending=False)

missing_summary

,missing_count,missing_percentage
medal,229959,85.254939
weight,61527,22.810504
height,58814,21.804687
age,9315,3.453441


In [23]:
# Investigating wether missing values are concentrated in particular years
missing_by_year = (
    athletes_df.groupby('year')
    .agg(
        missing_age=('age', lambda x: x.isna().sum()),
        missing_height=('height', lambda x: x.isna().sum()),
        missing_weight=('weight', lambda x: x.isna().sum()),
        total_records=('id', 'size')
    )
)

missing_by_year['missing_age_pct'] = (
    missing_by_year['missing_age'] / missing_by_year['total_records'] * 100
)

missing_by_year['missing_height_pct'] = (
    missing_by_year['missing_height'] / missing_by_year['total_records'] * 100
)

missing_by_year['missing_weight_pct'] = (
    missing_by_year['missing_weight'] / missing_by_year['total_records'] * 100
)

missing_by_year

,missing_age,missing_height,missing_weight,total_records,missing_age_pct,missing_height_pct,missing_weight_pct
year,,,,,,,
1896,163,334,331,380,42.894737,87.894737,87.105263
1900,770,1782,1819,1898,40.569020,93.888303,95.837724
1904,274,1088,1154,1301,21.060723,83.627978,88.700999
1906,743,1476,1528,1733,42.873630,85.170225,88.170802
1908,637,2595,2587,3069,20.755947,84.555230,84.294558
1912,156,3319,3444,4040,3.861386,82.153465,85.247525
1920,845,3525,3821,4292,19.687791,82.129543,89.026095
1924,1103,4602,4881,5570,19.802513,82.621185,87.630162
1928,924,4264,4521,5238,17.640321,81.405116,86.311569


#### Missing-value investigation:
 Missing age, height, and weight values are disproportionately concentrated in earlier Olympic Games. Height and weight are particularly incomplete before 1960, while age coverage improves substantially over time. Some later years also show increases in missing physical measurements. Missing values were preserved without imputation or row deletion to avoid introducing artificial data.

### 5) Categorical Values

In [24]:
categorical_cols = athletes_df.select_dtypes(include=['str', 'object']).columns

for col in categorical_cols:
    print(f"\n--- {col} ---")
    print("Unique values:", athletes_df[col].nunique())
    print(athletes_df[col].value_counts(dropna=False).head(10))


--- name ---
Unique values: 134732
name
Heikki Ilmari Savolainen     39
Joseph "Josy" Stoffel        38
Ioannis Theofilakis          36
Takashi Ono                  33
Alexandros Theofilakis       32
Andreas Wecker               32
Alfred August "Al" Jochim    31
Michel Mathiot               31
Johann "Hans" Sauter         31
Michael Fred Phelps, II      30
Name: count, dtype: int64

--- sex ---
Unique values: 2
sex
M    195353
F     74378
Name: count, dtype: int64

--- team ---
Unique values: 1184
team
United States    17598
France           11817
Great Britain    11264
Italy            10213
Germany           9230
Canada            9226
Japan             8269
Sweden            8004
Australia         7512
Hungary           6492
Name: count, dtype: int64

--- noc ---
Unique values: 230
noc
USA    18604
FRA    12551
GBR    12115
ITA    10668
GER     9734
CAN     9680
JPN     8424
SWE     8291
AUS     7637
HUN     6552
Name: count, dtype: int64

--- games ---
Unique values: 51
games
200

In [25]:
#investigate city names
sorted(athletes_df['city'].dropna().unique())

['Albertville',
 'Amsterdam',
 'Antwerpen',
 'Athina',
 'Atlanta',
 'Barcelona',
 'Beijing',
 'Berlin',
 'Calgary',
 'Chamonix',
 "Cortina d'Ampezzo",
 'Garmisch-Partenkirchen',
 'Grenoble',
 'Helsinki',
 'Innsbruck',
 'Lake Placid',
 'Lillehammer',
 'London',
 'Los Angeles',
 'Melbourne',
 'Mexico City',
 'Montreal',
 'Moskva',
 'Munich',
 'Nagano',
 'Oslo',
 'Paris',
 'Rio de Janeiro',
 'Roma',
 'Salt Lake City',
 'Sankt Moritz',
 'Sapporo',
 'Sarajevo',
 'Seoul',
 'Sochi',
 'Squaw Valley',
 'St. Louis',
 'Stockholm',
 'Sydney',
 'Tokyo',
 'Torino',
 'Vancouver']

In [26]:
for city in sorted(athletes_df['city'].dropna().unique()):
    if city != city.strip():
        print(repr(city))

In [27]:
#Investigate sport names
sorted(athletes_df['sport'].dropna().unique())

['Aeronautics',
 'Alpine Skiing',
 'Alpinism',
 'Archery',
 'Art Competitions',
 'Athletics',
 'Badminton',
 'Baseball',
 'Basketball',
 'Basque Pelota',
 'Beach Volleyball',
 'Biathlon',
 'Bobsleigh',
 'Boxing',
 'Canoeing',
 'Cricket',
 'Croquet',
 'Cross Country Skiing',
 'Curling',
 'Cycling',
 'Diving',
 'Equestrianism',
 'Fencing',
 'Figure Skating',
 'Football',
 'Freestyle Skiing',
 'Golf',
 'Gymnastics',
 'Handball',
 'Hockey',
 'Ice Hockey',
 'Jeu De Paume',
 'Judo',
 'Lacrosse',
 'Luge',
 'Military Ski Patrol',
 'Modern Pentathlon',
 'Motorboating',
 'Nordic Combined',
 'Polo',
 'Racquets',
 'Rhythmic Gymnastics',
 'Roque',
 'Rowing',
 'Rugby',
 'Rugby Sevens',
 'Sailing',
 'Shooting',
 'Short Track Speed Skating',
 'Skeleton',
 'Ski Jumping',
 'Snowboarding',
 'Softball',
 'Speed Skating',
 'Swimming',
 'Synchronized Swimming',
 'Table Tennis',
 'Taekwondo',
 'Tennis',
 'Trampolining',
 'Triathlon',
 'Tug-Of-War',
 'Volleyball',
 'Water Polo',
 'Weightlifting',
 'Wrestling'

In [28]:
athlete_nocs = set(athletes_df['noc'].dropna().unique())
reference_nocs = set(noc_df['NOC'].dropna().unique())

unmatched_nocs = athlete_nocs - reference_nocs

unmatched_nocs

{'SGP'}

In [29]:
noc_df[noc_df['NOC'].str.contains('SGP', na=False)]

,NOC,region,notes


In [30]:
noc_df[
    noc_df['region'].str.contains('Singapore', case=False, na=False)
]

,NOC,region,notes
178,SIN,Singapore,NaN


In [31]:
athletes_df.loc[athletes_df['noc'] == 'SGP', 'noc'].value_counts()

noc
SGP    349
Name: count, dtype: int64

In [32]:
athletes_df['noc'] = athletes_df['noc'].replace('SGP', 'SIN')

In [33]:
athletes_df['noc'].isin(noc_df['NOC']).all()

np.True_

#### NOC validation:
 One unmatched NOC code, SGP, was identified. Investigation showed that the reference dataset uses SIN for Singapore. The code was standardized from SGP to SIN to ensure consistency when merging the datasets.

### 6) Merge NOC reference data

In [34]:
noc_df['NOC'].duplicated().sum()

np.int64(0)

In [36]:
noc_df[noc_df['NOC'].isin(['ROT', 'TUV', 'UNK'])]

,NOC,region,notes
168,ROT,NaN,Refugee Olympic Team
208,TUV,NaN,Tuvalu
213,UNK,NaN,Unknown


In [37]:
noc_df.loc[noc_df['NOC'] == 'ROT', 'region'] = 'Refugee Olympic Team'
noc_df.loc[noc_df['NOC'] == 'TUV', 'region'] = 'Tuvalu'

In [38]:
noc_df[noc_df['NOC'].isin(['ROT', 'TUV', 'UNK'])]

,NOC,region,notes
168,ROT,Refugee Olympic Team,Refugee Olympic Team
208,TUV,Tuvalu,Tuvalu
213,UNK,NaN,Unknown


In [39]:
rows_before = len(athletes_df)

In [40]:
athletes_df = athletes_df.merge(
    noc_df,
    left_on='noc',
    right_on='NOC',
    how='left',
    validate='many_to_one'
)

In [41]:
athletes_df = athletes_df.drop(columns='NOC')

In [42]:
print("Original rows:", rows_before)
print("Rows after merge:", len(athletes_df))
print("Missing regions:", athletes_df['region'].isna().sum())

Original rows: 269731
Rows after merge: 269731
Missing regions: 2


In [43]:
athletes_df.loc[
    athletes_df['region'].isna(),
    ['noc', 'team', 'games']
].drop_duplicates()

,noc,team,games
60733,UNK,Unknown,1912 Summer


### 7) Numerical values validation

In [44]:
athletes_df[['age', 'height', 'weight', 'year']].describe().T

,count,mean,std,min,25%,50%,75%,max
age,260416.0,25.454776,6.163869,10.0,21.0,24.0,28.0,97.0
height,210917.0,175.338953,10.518507,127.0,168.0,175.0,183.0,226.0
weight,208204.0,70.701778,14.349027,25.0,60.0,70.0,79.0,214.0
year,269731.0,1978.623073,29.752055,1896.0,1960.0,1988.0,2002.0,2016.0


Numerical values were inspected for age, height, and weight. Extreme height and weight values, as well as the minimum age, were investigated and found to be valid. The maximum age of 97 was identified in historical Art Competition records involving artists whose works were submitted after their deaths. These records were preserved, as the unusual ages reflect historical data conventions rather than ordinary athlete ages.

### 8) Final quality validation

In [45]:
print("Dataset shape:", athletes_df.shape)

print(
    "Exact duplicate rows:",
    athletes_df.duplicated().sum()
)

print(
    "Duplicate column names:",
    athletes_df.columns.duplicated().sum()
)

Dataset shape: (269731, 17)
Exact duplicate rows: 0
Duplicate column names: 0


In [46]:
missing_summary = pd.DataFrame({
    'missing_count': athletes_df.isna().sum(),
    'missing_percentage': athletes_df.isna().mean() * 100
})

missing_summary = missing_summary[
    missing_summary['missing_count'] > 0
].sort_values('missing_percentage', ascending=False)

missing_summary

,missing_count,missing_percentage
notes,264696,98.133325
medal,229959,85.254939
weight,61527,22.810504
height,58814,21.804687
age,9315,3.453441
region,2,0.000741


In [47]:
athletes_df.groupby('noc')['region'].apply(
    lambda x: x.isna().all()
).loc[lambda x: x].index.tolist()

['UNK']

In [48]:
print("Data types:")
display(athletes_df.dtypes)

print("\nMissing values in essential columns:")
essential_cols = [
    'id', 'name', 'sex', 'noc',
    'games', 'season', 'sport', 'event'
]

print(athletes_df[essential_cols].isna().sum())

Data types:


id          int64
name          str
sex           str
age       float64
height    float64
weight    float64
team          str
noc           str
games         str
year        int64
season        str
city          str
sport         str
event         str
medal         str
region        str
notes         str
dtype: object


Missing values in essential columns:
id        0
name      0
sex       0
noc       0
games     0
season    0
sport     0
event     0
dtype: int64


### Final Data Quality Validation

The cleaned dataset underwent final validation to ensure data integrity and consistency.

- Confirmed that no exact duplicate rows remain.
- Confirmed that all column names are unique.
- Verified that all essential identification columns contain no missing values.
- Confirmed that NOC codes are correctly mapped, with only the intentionally unidentified `UNK` delegation remaining without a region.
- Verified that missing-value counts are consistent with the earlier investigation.
- Investigated extreme numerical values and preserved valid historical records.

The final dataset contains **269,731 rows and 17 columns**. Missing values were preserved where appropriate rather than removed or imputed without justification.